In [ ]:
!pip install -U langchain langchain-community langchain-google-genai langchain-chroma chromadb tiktoken pypdf

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your API Key: ")

Enter your API Key: ··········


In [ ]:
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [ ]:
from langchain_core.documents import Document

documents = [
    # HR Department
    Document(
        page_content="""
        Employees are entitled to 20 annual leave days per year.
        Leave requests must be submitted to the HR department
        at least 3 days in advance.
        """,
        metadata={"source": "leave_policy", "department": "HR"}
    ),

    Document(
        page_content="""
        New employees must complete a three-month probation period.
        During this period, their performance and work quality
        are evaluated by their managers.
        """,
        metadata={"source": "probation_policy", "department": "HR"}
    ),

    # IT Department
    Document(
        page_content="""
        Employees must use strong passwords containing at least
        12 characters, including uppercase letters, lowercase letters,
        numbers, and special characters.
        Passwords must be changed every 90 days.
        """,
        metadata={"source": "security_policy", "department": "IT"}
    ),

    Document(
        page_content="""
        Employees can request technical support by submitting
        a ticket through the IT helpdesk.
        Critical technical issues are prioritized and should
        receive an initial response within 2 hours.
        """,
        metadata={"source": "technical_support", "department": "IT"}
    ),

    # Finance Department
    Document(
        page_content="""
        Employees must submit their monthly expense reports
        before the 5th day of the following month.
        All expenses must include valid receipts for reimbursement.
        """,
        metadata={"source": "expense_policy", "department": "Finance"}
    )
]

In [ ]:
vector_store = Chroma(
    embedding_function=embedding_model,
    persist_directory="chroma_db" ,
    collection_name="documents"
)

In [ ]:
# Add Documents
vector_store.add_documents(documents)

['ccfe592d-cf20-4075-87c1-06ff5b717922',
 'c3d7c6c3-75d3-4496-9793-e88aee5ed615',
 '7534649b-e9bb-491a-8d37-b37af72d5950',
 '2bf65d44-33fb-4906-9995-e45fd9dfe3e1',
 '641c5fc5-9f63-4561-9945-da8712d9772f']

In [ ]:
# View Documents
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['ccfe592d-cf20-4075-87c1-06ff5b717922',
  'c3d7c6c3-75d3-4496-9793-e88aee5ed615',
  '7534649b-e9bb-491a-8d37-b37af72d5950',
  '2bf65d44-33fb-4906-9995-e45fd9dfe3e1',
  '641c5fc5-9f63-4561-9945-da8712d9772f'],
 'embeddings': array([[-0.02936229,  0.02927721,  0.00414369, ...,  0.00732067,
          0.01884518, -0.00849293],
        [-0.02917701,  0.02918086,  0.01259549, ...,  0.00376472,
          0.00856969, -0.01322312],
        [-0.01675636,  0.0374884 ,  0.00078233, ..., -0.00467031,
          0.00552703, -0.01648717],
        [-0.00581378,  0.02437568,  0.00038191, ...,  0.00595007,
          0.01400304, -0.02454454],
        [-0.02796465,  0.03756307,  0.01265445, ...,  0.00012998,
          0.01245027, -0.01726962]], shape=(5, 3072)),
 'documents': ['\n        Employees are entitled to 20 annual leave days per year.\n        Leave requests must be submitted to the HR department\n        at least 3 days in advance.\n        ',
  '\n        New employees must complete a t

In [ ]:
# Search Documents
vector_store.similarity_search(
    query="How long is the probation period?",
    k=1
)

[Document(id='c3d7c6c3-75d3-4496-9793-e88aee5ed615', metadata={'source': 'probation_policy', 'department': 'HR'}, page_content='\n        New employees must complete a three-month probation period.\n        During this period, their performance and work quality\n        are evaluated by their managers.\n        ')]

In [ ]:
# Search with Similarity Score
vector_store.similarity_search_with_score(
    query="How long is the probation period?",
    k=2
)

[(Document(id='c3d7c6c3-75d3-4496-9793-e88aee5ed615', metadata={'source': 'probation_policy', 'department': 'HR'}, page_content='\n        New employees must complete a three-month probation period.\n        During this period, their performance and work quality\n        are evaluated by their managers.\n        '),
  0.4995773136615753),
 (Document(id='ccfe592d-cf20-4075-87c1-06ff5b717922', metadata={'source': 'leave_policy', 'department': 'HR'}, page_content='\n        Employees are entitled to 20 annual leave days per year.\n        Leave requests must be submitted to the HR department\n        at least 3 days in advance.\n        '),
  0.7463481426239014)]

In [ ]:
# Metadata Filtering
vector_store.similarity_search_with_score(
    query="How many annual leave days are employees entitled to?",
    filter={"department": "HR"}
)

[(Document(id='ccfe592d-cf20-4075-87c1-06ff5b717922', metadata={'source': 'leave_policy', 'department': 'HR'}, page_content='\n        Employees are entitled to 20 annual leave days per year.\n        Leave requests must be submitted to the HR department\n        at least 3 days in advance.\n        '),
  0.5108032822608948),
 (Document(id='c3d7c6c3-75d3-4496-9793-e88aee5ed615', metadata={'source': 'probation_policy', 'department': 'HR'}, page_content='\n        New employees must complete a three-month probation period.\n        During this period, their performance and work quality\n        are evaluated by their managers.\n        '),
  0.8155310153961182)]

In [ ]:
updated_doc = Document(
    page_content="""
    Employees are entitled to 35 annual leave days per year.
    Leave requests must be submitted to the HR department
    at least 3 days in advance.
    """,
    metadata={
        "source": "leave_policy",
        "department": "HR"
    }
)

In [ ]:
vector_store.update_document(
    document_id='ccfe592d-cf20-4075-87c1-06ff5b717922',
    document=updated_doc
)

In [ ]:
# View Documents
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['ccfe592d-cf20-4075-87c1-06ff5b717922',
  'c3d7c6c3-75d3-4496-9793-e88aee5ed615',
  '7534649b-e9bb-491a-8d37-b37af72d5950',
  '2bf65d44-33fb-4906-9995-e45fd9dfe3e1',
  '641c5fc5-9f63-4561-9945-da8712d9772f'],
 'embeddings': array([[-0.03768641,  0.02809716,  0.00573877, ...,  0.00474947,
          0.02152687, -0.01068038],
        [-0.02917701,  0.02918086,  0.01259549, ...,  0.00376472,
          0.00856969, -0.01322312],
        [-0.01675636,  0.0374884 ,  0.00078233, ..., -0.00467031,
          0.00552703, -0.01648717],
        [-0.00581378,  0.02437568,  0.00038191, ...,  0.00595007,
          0.01400304, -0.02454454],
        [-0.02796465,  0.03756307,  0.01265445, ...,  0.00012998,
          0.01245027, -0.01726962]], shape=(5, 3072)),
 'documents': ['\n    Employees are entitled to 35 annual leave days per year.\n    Leave requests must be submitted to the HR department\n    at least 3 days in advance.\n    ',
  '\n        New employees must complete a three-month proba

In [ ]:
# Delete Document
vector_store.delete(ids =['ccfe592d-cf20-4075-87c1-06ff5b717922'])

In [ ]:
# View Documents
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['c3d7c6c3-75d3-4496-9793-e88aee5ed615',
  '7534649b-e9bb-491a-8d37-b37af72d5950',
  '2bf65d44-33fb-4906-9995-e45fd9dfe3e1',
  '641c5fc5-9f63-4561-9945-da8712d9772f'],
 'embeddings': array([[-0.02917701,  0.02918086,  0.01259549, ...,  0.00376472,
          0.00856969, -0.01322312],
        [-0.01675636,  0.0374884 ,  0.00078233, ..., -0.00467031,
          0.00552703, -0.01648717],
        [-0.00581378,  0.02437568,  0.00038191, ...,  0.00595007,
          0.01400304, -0.02454454],
        [-0.02796465,  0.03756307,  0.01265445, ...,  0.00012998,
          0.01245027, -0.01726962]], shape=(4, 3072)),
 'documents': ['\n        New employees must complete a three-month probation period.\n        During this period, their performance and work quality\n        are evaluated by their managers.\n        ',
  '\n        Employees must use strong passwords containing at least\n        12 characters, including uppercase letters, lowercase letters,\n        numbers, and special characte